# End to end: 10 videos → feature CSV → Loss-1 training

Clones **[GazeVLM-HWSW-Codesign](https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign)**
and runs the repo's own scripts — no code is redefined here, so what you verify is what
is actually in the repo.

```
python -m src.dataprep.build_feature_csv   # 10 videos -> CSV + .npz features
python -m src.loss1.train                  # InfoNCE pretraining
```

## What is being verified

**The headline: does the Loss-1 loss actually go down?**

But a falling training loss on its own proves very little — a model with ~1.5 M parameters
and a few hundred rows can simply memorise. So three things are checked, and all three
have to hold:

| # | Check | Why it matters |
|---|---|---|
| 1 | **train loss falls** | necessary, but memorisation produces this too |
| 2 | **val top-1 beats chance** on held-out **videos** | the model generalises to wearers and scenes it never saw |
| 3 | **the shuffle control does NOT** beat chance | rules out the result being an artefact of the setup |

Check 3 is the one that makes checks 1 and 2 mean something. `--shuffle_control` pairs each
frame pair with **another row's** gaze rates, destroying the correspondence the loss is
supposed to learn. If that run also succeeds, the success was never about gaze.

## Cost

10 videos ≈ **25 GB** of downloads (the VRS dominates, and is needed for the calibration
projection). `--cleanup_raw` deletes each video's raw files once its features are written,
so peak disk stays near one video. Budget **15–25 min** for the build, a couple of minutes
for each training run.

**No GPU required.**

## 1 — Clone the repo and install dependencies

In [ ]:
!git clone -q https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git /content/GazeVLM
%cd /content/GazeVLM
!git log --oneline -3

# requirements.txt covers the pipeline; these two are needed on top
!pip -q install -r requirements.txt
!pip -q install projectaria-tools matplotlib

import os, json, time
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

print("\nrepo:", os.getcwd())
print("torch", torch.__version__, "| device", "cuda" if torch.cuda.is_available() else "cpu")
!ls src

## 2 — Upload the download-links JSON

Not in the repo — it holds signed URLs tied to your Aria account, and they expire after
~14 days. Re-download from
[projectaria.com/datasets/aea](https://www.projectaria.com/datasets/aea/) if the build
fails on a download.

In [ ]:
from google.colab import files
up = files.upload()                    # pick your aea_download_urls.json
URLS_JSON = "/content/" + list(up.keys())[0]
os.rename(list(up.keys())[0], URLS_JSON)

# or from Drive:
# from google.colab import drive; drive.mount('/content/drive')
# URLS_JSON = "/content/drive/MyDrive/aea/aea_download_urls.json"

meta = json.load(open(URLS_JSON))
print(f"{len(meta['sequences'])} videos available   ->  {URLS_JSON}")

## 3 — Build the dataset: 10 videos

Runs `src/dataprep/build_feature_csv.py` from the clone. This is the slow cell.

Per video: download → 1 FPS frames → gaze projection → DINOv2 encode → one `.npz` per
frame → one CSV row per consecutive frame pair. Expect **~89 rows per video**, fewer for
videos shorter than `--max_seconds`.

In [ ]:
## 3 — Build the dataset: 10 videos, full length

Runs `src/dataprep/build_feature_csv.py` from the clone. This is the slow cell.

Per video: download → 1 FPS frames → gaze projection → DINOv2 encode → one `.npz` per
frame → one CSV row per consecutive frame pair.

`MAX_SECONDS = 0` uses the **whole** video. AEA sequences average ~193 s (range 67–456 s),
so expect **~190 rows per video** and **~1,900 rows** in total. Capping this would throw
away footage the download already paid for.

N_VIDEOS    = 10
SEED        = 0
MAX_SECONDS = 0           # 0 = the WHOLE video. Set a number only to trim a debug run.
OUT_CSV     = "/content/data/feature_dataset.csv"

t0 = time.time()
!python -m src.dataprep.build_feature_csv \
    --urls_json  "{URLS_JSON}" \
    --out_csv    "{OUT_CSV}" \
    --raw_dir    /content/data/raw \
    --frames_dir /content/data/frames_1fps \
    --feat_dir   /content/data/features \
    --n_videos   {N_VIDEOS} \
    --seed       {SEED} \
    --max_seconds {MAX_SECONDS} \
    --cleanup_raw

print(f"\nbuild took {(time.time()-t0)/60:.1f} min")

In [ ]:
df = pd.read_csv(OUT_CSV)
print(f"{len(df)} rows x {df.shape[1]} columns from {df['sequence'].nunique()} videos\n")

show = df.copy()
show["feat_frame_1"] = show["feat_frame_1"].apply(os.path.basename)
show["feat_frame_2"] = show["feat_frame_2"].apply(os.path.basename)
show["gaze_rates_window"] = show["gaze_rates_window"].str.slice(0, 28) + " ..."
display(show.head(6))

print("rows per video:")
display(df.groupby("sequence").size().rename("rows").to_frame())

print("labels:")
display(df[["frame_similarity", "gaze_patch_token_sim", "n_velocities"]].describe().round(3))

feat_n = sum(len(fs) for _, _, fs in os.walk("/content/data/features"))
print(f"\nfeature files on disk: {feat_n}")
print(f"train/val split will be by VIDEO: ~{int(df['sequence'].nunique()*0.8)} train / "
      f"{df['sequence'].nunique() - int(df['sequence'].nunique()*0.8)} val")

## 5 — Train Loss 1

Runs `src/loss1/train.py`. Defaults are EgoDistill §4.1's pretraining settings: AdamW,
lr 1e-4, batch 64, 50 epochs, τ = 0.1.

Watch the printed `top1` against `chance`. Loss starting near **ln(batch) ≈ 4.16** is
correct for InfoNCE at initialisation.

In [ ]:
EPOCHS = 50
BATCH  = 64
RUN    = "/content/runs/loss1"

t0 = time.time()
!python -m src.loss1.train \
    --csv     "{OUT_CSV}" \
    --out_dir "{RUN}" \
    --epochs  {EPOCHS} \
    --batch_size {BATCH} \
    --val_frac 0.2 \
    --seed 0

print(f"\ntraining took {(time.time()-t0)/60:.1f} min")

## 6 — THE VERIFICATION: does the loss actually go down?

Reads `history.json` and plots it. Three panels: the loss curve, retrieval accuracy
against chance, and the train/val gap.

In [ ]:
def load_history(path):
    h = json.load(open(os.path.join(path, "history.json")))
    ep  = [r["epoch"] for r in h]
    tr  = [r["train"]["loss"] for r in h]
    t1  = [r["train"]["top1"] for r in h]
    ch  = [r["train"]["chance"] for r in h]
    has_val = h[0]["val"] is not None
    vl  = [r["val"]["loss"] for r in h] if has_val else None
    v1  = [r["val"]["top1"] for r in h] if has_val else None
    return dict(ep=ep, tr=tr, t1=t1, ch=ch, vl=vl, v1=v1)


H = load_history(RUN)
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))

ax[0].plot(H["ep"], H["tr"], lw=2, label="train")
if H["vl"]:
    ax[0].plot(H["ep"], H["vl"], lw=2, label="val")
ax[0].axhline(np.log(BATCH), ls="--", c="k", lw=1,
              label=f"random = ln({BATCH}) = {np.log(BATCH):.2f}")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("InfoNCE loss")
ax[0].set_title("DOES THE LOSS GO DOWN?", fontweight="bold"); ax[0].legend()

ax[1].plot(H["ep"], 100*np.array(H["t1"]), lw=2, label="train top-1")
if H["v1"]:
    ax[1].plot(H["ep"], 100*np.array(H["v1"]), lw=2, label="val top-1")
ax[1].axhline(100*np.mean(H["ch"]), ls="--", c="k", lw=1,
              label=f"chance = {100*np.mean(H['ch']):.1f}%")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("retrieval top-1 (%)")
ax[1].set_title("can it find the right gaze signal?"); ax[1].legend()

if H["vl"]:
    ax[2].plot(H["ep"], np.array(H["vl"]) - np.array(H["tr"]), lw=2, c="tab:red")
    ax[2].axhline(0, ls="--", c="k", lw=1)
    ax[2].set_xlabel("epoch"); ax[2].set_ylabel("val loss - train loss")
    ax[2].set_title("overfitting gap (rising = memorising)")
else:
    ax[2].axis("off")
plt.tight_layout(); plt.show()

print(f"train loss : {H['tr'][0]:.4f}  ->  {H['tr'][-1]:.4f}   "
      f"({100*(1-H['tr'][-1]/H['tr'][0]):+.1f}%)")
if H["vl"]:
    print(f"val   loss : {H['vl'][0]:.4f}  ->  {H['vl'][-1]:.4f}")
    print(f"val   top-1: {100*max(H['v1']):.1f}% best   vs chance {100*np.mean(H['ch']):.1f}%")
print(f"train top-1: {100*max(H['t1']):.1f}% best")

## 7 — The control run

Identical settings, one change: `--shuffle_control` pairs each frame pair with **another
row's** gaze rates. The correspondence InfoNCE is meant to learn no longer exists.

**This run should fail.** Loss should sit near ln(batch), top-1 near chance. If it learns
anyway, then the real run's result came from something other than gaze — and the whole
experiment needs rethinking before any conclusion is drawn.

In [ ]:
RUN_CTRL = "/content/runs/loss1_control"

!python -m src.loss1.train \
    --csv     "{OUT_CSV}" \
    --out_dir "{RUN_CTRL}" \
    --epochs  {EPOCHS} \
    --batch_size {BATCH} \
    --val_frac 0.2 \
    --seed 0 \
    --shuffle_control

## 8 — Side by side, and the verdict

In [ ]:
C = load_history(RUN_CTRL)
chance = float(np.mean(H["ch"]))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
ax[0].plot(H["ep"], H["tr"], lw=2, c="tab:blue",   label="real: train")
ax[0].plot(C["ep"], C["tr"], lw=2, c="tab:red", ls="--", label="control: train")
ax[0].axhline(np.log(BATCH), ls=":", c="k", lw=1, label=f"random = {np.log(BATCH):.2f}")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("InfoNCE loss")
ax[0].set_title("real vs shuffled gaze", fontweight="bold"); ax[0].legend()

if H["v1"] and C["v1"]:
    ax[1].plot(H["ep"], 100*np.array(H["v1"]), lw=2, c="tab:blue", label="real: val top-1")
    ax[1].plot(C["ep"], 100*np.array(C["v1"]), lw=2, c="tab:red", ls="--", label="control: val top-1")
ax[1].axhline(100*chance, ls=":", c="k", lw=1, label=f"chance = {100*chance:.1f}%")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("val top-1 (%)")
ax[1].set_title("generalisation to unseen videos"); ax[1].legend()
plt.tight_layout(); plt.show()

# ---------------------------------------------------------------- verdict
loss_drop   = 1 - H["tr"][-1] / H["tr"][0]
val_top1    = max(H["v1"]) if H["v1"] else float("nan")
ctrl_top1   = max(C["v1"]) if C["v1"] else float("nan")

checks = [
    ("1. train loss falls (>10%)",              loss_drop > 0.10,
     f"{H['tr'][0]:.3f} -> {H['tr'][-1]:.3f}  ({100*loss_drop:+.1f}%)"),
    ("2. val top-1 beats chance (>2x)",         val_top1 > 2*chance,
     f"{100*val_top1:.1f}%  vs chance {100*chance:.1f}%"),
    ("3. control does NOT beat chance (<2x)",   ctrl_top1 < 2*chance,
     f"{100*ctrl_top1:.1f}%  vs chance {100*chance:.1f}%"),
]

print("=" * 74)
for name, ok, detail in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}]  {name:36s} {detail}")
print("=" * 74)

if all(c[1] for c in checks):
    print("\n  ALL THREE HOLD.")
    print("  The loss falls, it generalises to videos never seen in training, and it")
    print("  does NOT fall when the gaze is shuffled. That is real evidence that eye")
    print("  motion carries information about how the picture changed.")
elif checks[0][1] and not checks[2][1]:
    print("\n  LOSS FELL IN BOTH RUNS -- so it is not learning from gaze.")
    print("  The model is exploiting something else: batch composition, a leaky split,")
    print("  or memorisation. Do not report the real run as a positive result.")
elif checks[0][1] and not checks[1][1]:
    print("\n  TRAINS BUT DOES NOT GENERALISE -- it is memorising the training videos.")
    print("  Expected with only a handful of videos. Raise --n_videos and --max_seconds.")
else:
    print("\n  THE LOSS DID NOT FALL. Before concluding the premise is wrong, check:")
    print("  the dataset size, the learning rate, and that 9 steps per window is")
    print("  enough signal (EgoDistill used 422 IMU samples per clip).")

---

## How to read the outcome

| Real run | Control run | Meaning |
|---|---|---|
| loss falls, val beats chance | stays at chance | **the premise holds** on this data |
| loss falls, val at chance | stays at chance | learning, but memorising — needs more videos |
| loss falls | **also falls** | not learning from gaze; an artefact of the setup |
| loss flat | flat | no signal found — see the checklist below |

## If the loss does not fall

Things to try, roughly in order of likely impact:

1. **More data.** 10 videos at 90 s gives ~890 rows. EgoDistill pretrained on thousands of
   clips. Raise `--n_videos`, and set `--max_seconds 0` for whole videos (~212 rows each).
2. **Longer windows.** Each row carries only **9** gaze steps; EgoDistill's IMU tensor was
   422, and this project's own architecture slide specifies 200×3. Pairing frames *t* and
   *t+K* rather than *t* and *t+1* would give `10K−1` steps.
3. **Batch size.** InfoNCE takes its negatives from the batch, so a bigger batch is a
   harder and more informative task — if there are enough rows to support it.

## A caveat about batch composition

`VideoBalancedBatchSampler` caps how many rows from one video share a batch, because two
windows seconds apart in the same video can have near-identical gaze rates — false
negatives the loss would actively push apart. With few videos this cap binds hard. If
check 2 fails while check 1 passes, more videos is the first thing to try, not a different
architecture.